# Assignment 1 - Computational Social Science (02467)
##### Group 3: Andrew Osias (s223719) & Mia Paarup (s213304)
https://github.com/aro2016y/comsocsci-assignment1/upload/main

### Part 1: Ready Made vs Custom Made Data

A study by Centola was undertaken to determine the social contagion when comparing different type of network clusterings. 

This type of study highlights the difference between Ready Made vs Custom Made Data, particularly regarding how tight control allows for more robust results, but less generalizability when accounting for real world effects. This is because the fact that it is so tightly designed, means that it's easier to isolate points of interest, but you lose the fact that the real world is often messier and realism is limited.

Nicolaides & Aral conversely use fitness tracker and weather data from 1.1M runners to isolate points of interest in the real world. The pros and cons are almost opposites in that effects are harder to isolate. All studies suffer form limitations, and it is in fact in the metaanalyses of such studies that more general patterns are much more defensible and scientifically valid.

Both of these are perfect examples of custom vs ready-made data. You either look for strong internal validity at the cost of realism, or go for realism, but are more limited by the messiness of such complex systems and more biased for interpretation and choice of analysis - more inference. 


### Part 2: Finding Researchers using the OpenAlex API

In this section, we retrieve a list of authors from the IC2S2 conference by scraping HTML data. First we import the necessary libraries as well as the URL.

In [15]:
from bs4 import BeautifulSoup  # A package to work with HTML data
import requests               # A package to make HTTP requests
LINK = "https://ic2s2-2025.org/program/"
r = requests.get(LINK)
soup = BeautifulSoup(r.content)
ems = soup.select("#schedule-list ol li em")
import pandas as pd

url = "https://ic2s2-2025.org/files/ic2s2_2025_schedule_v5.csv"
df = pd.read_csv(url)

df.columns

Index(['date_start', 'date_end', 'type', 'title', 'abstract', 'author',
       'location', 'session', 'session_order', 'chair', 'openreview_id',
       'notes'],
      dtype='object')

A list of chairs for the conference are found first.

In [16]:
# chairs
chairs = (
    df["chair"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("")]
    .drop_duplicates()
    .reset_index(drop=True)
)

chairs_df = chairs.to_frame("Chair")
chairs_df

,Chair
0,Sonia Yeh
1,David Garcia
2,Étienne Ollion
3,Hendrik Erz
4,Taha Yasseri
5,Mirta Galesic
6,Friedolin Merhout
7,Adel Daoud
8,Lorien Jasny
9,Sebastian Stier


We then use authors as our second source of data.

In [17]:
authors_col = "author"

author = (
    df[authors_col]
    .dropna()
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
    .loc[lambda s: s.ne("")]
    .drop_duplicates()
    .reset_index(drop=True)
)

authors_df = author.to_frame("Author")
authors_df

,Author
0,Étienne Ollion
1,Émilien Schultz
2,Miriam Schirmer
3,Julia Mendelsohn
4,Dustin Wright
...,...
1560,Petter Holme
1561,Christian Steglich
1562,Neha Gondal
1563,Victor Møller Poulsen


In [19]:
combined = pd.concat([authors_df, chairs_df], ignore_index=True).drop_duplicates()

combined.to_csv("authors_list.csv", index=False, encoding="utf-8")

This exercise went quite smoothly, due to the small dataset, and very limited scope of analysis. Most of the effort went into understanding how HTML works, as well as API's. The implementation of this exercise was done quite a while ago, however the most pressing issues (to choose between very minor issues) was simply understanding where to look on the website, when inspecting, which window to look at, as the developer tools are much more complex than what our purposes required. ChatGPT was a massive help for navigating these developer tools as for what windows are relevant and for explanations into how the data is fed into the browser, which led to realizing that much of the data was saved in files, that can be opened in a new tab - a very helpful viewer, although memory is foggy due to this being implemented quite some time ago. 

### Part 3: Collecting Research Articles

In [24]:
print(response.json())

{'error': 'Rate limit exceeded', 'message': 'Insufficient budget. This request costs $0.001 but you only have $0.0008 remaining. Resets at midnight UTC. Need more? Add funds at https://openalex.org/pricing', 'retryAfter': 11359, 'costUsd': 0.001, 'dailyRemainingUsd': 0.0008, 'prepaidRemainingUsd': 0, 'creditsRequired': 10, 'creditsRemaining': 8, 'onetimeCreditsRemaining': 0}


In [25]:
import requests
import seaborn as sns
import pandas as pd
#Extract CSV of author name list made through ComSoc1
authors_df = pd.read_csv("authors_list.csv")
names = authors_df["Author"].tolist()


#loop through names and get a row of relevant results
url = "https://api.openalex.org/authors"
rows = []
for name in names:
    params = {
        "search": name, "per_page": 1, "api_key": "A9R2odh6HKkiizaS7I00sm"}
    #EXTRACT API CALL HERE
    response = requests.get(url, params=params)

    data = response.json()
    
    
    
    if len(data["results"]) == 0:
        continue
    
    author = data["results"][0]

    row = {
    "id": author["id"],
    "display_name": author["display_name"],
    "works_api_url": author["works_api_url"],
    "h_index": author["summary_stats"]["h_index"],
    "works_count": author["works_count"],
    "country_code": (
        author["last_known_institution"]["country_code"]
        if author.get("last_known_institution")
        else None
    )
    }
    rows.append(row)

final_df = pd.DataFrame(rows)
final_df

KeyError: 'results'

In [ ]:
final_df.to_csv("df1.csv", index=False, encoding="utf-8")

In [26]:
import requests
import seaborn as sns
import pandas as pd
from joblib import Parallel, delayed
#Extract CSV of authors data list made through ComSoc2
authors_df = pd.read_csv("df1.csv")
authors_df

,id,display_name,works_api_url,h_index,works_count,country_code
0,https://openalex.org/A5022704573,Étienne Ollion,https://api.openalex.org/works?filter=author.i...,17,131,NaN
1,https://openalex.org/A5079294298,Émilien Schultz,https://api.openalex.org/works?filter=author.i...,11,114,NaN
2,https://openalex.org/A5036764962,Miriam Schirmer,https://api.openalex.org/works?filter=author.i...,3,13,NaN
3,https://openalex.org/A5038833789,Julia Mendelsohn,https://api.openalex.org/works?filter=author.i...,9,23,NaN
4,https://openalex.org/A5090577017,Dustin E. Wright,https://api.openalex.org/works?filter=author.i...,9,25,NaN
...,...,...,...,...,...,...
1485,https://openalex.org/A5079517520,Petter Holme,https://api.openalex.org/works?filter=author.i...,51,251,NaN
1486,https://openalex.org/A5030685996,Christian Steglich,https://api.openalex.org/works?filter=author.i...,42,104,NaN
1487,https://openalex.org/A5034089559,Neha Gondal,https://api.openalex.org/works?filter=author.i...,11,24,NaN
1488,https://openalex.org/A5059830287,Victor Møller Poulsen,https://api.openalex.org/works?filter=author.i...,2,7,NaN


In [35]:
#loop through names and get a row of relevant results
url = "https://api.openalex.org/works"
def fetch_author(row):
    rows = []
    name = row["id"]
    works_count = row["works_count"]

    if not (5 < works_count < 5000):
        return []


    short_id = name.split("/")[-1] 
    filter_string = (
        f"author.id:{short_id},"
        f"cited_by_count:>10,"
        f"authors_count:<10"
    )
    
    cursor = "*"

    while cursor:
        params = {
            "filter": filter_string,
            "select": "id,publication_year,cited_by_count,authorships,title,abstract_inverted_index,concepts",
            "per_page": 200,
            "cursor": cursor,
            "api_key": "A9R2odh6HKkiizaS7I00sm"
        }

        response = requests.get(url, params=params)
        data = response.json()

        if "results" not in data:
            break

        for work in data["results"]:
            work_row = {
                "id": work["id"],
                "publication_year": work["publication_year"],
                "cited_by_count": work["cited_by_count"],
                "author_ids": [a["author"]["id"].split("/")[-1] for a in work["authorships"] if a["author"]["id"]],
                "title": work["title"],
                "abstract_inverted_index": work["abstract_inverted_index"],
                "concepts": [c["display_name"] for c in work["concepts"] if c["level"] == 0]
            }
            rows.append(work_row)

        cursor = data["meta"]["next_cursor"]

    return rows
# ---- Parallel execution ----

results = Parallel(n_jobs=25)(
    delayed(fetch_author)(row)
    for _, row in authors_df.iterrows()
)

# flatten list of lists
rows = [item for sublist in results for item in sublist]

temp_df = pd.DataFrame(rows)
temp_df

""


In [36]:
social_science = ["Sociology", "Psychology", "Economics", "Political Science"]
quantitative = ["Mathematics", "Physics", "Computer Science"]

temp2_df = temp_df[
    temp_df["concepts"].apply(
        lambda conditions: any(item in conditions for item in social_science) and
                           any(item in conditions for item in quantitative)
    )
]
df2 = temp2_df[["id","publication_year","cited_by_count","author_ids"]]
df3 = temp2_df[["id","title","abstract_inverted_index"]]
df2

KeyError: 'concepts'

In [31]:
df3

,id,title,abstract_inverted_index
0,https://openalex.org/W2159141474,The Superiority of Economists,"{'In': [0], 'this': [1, 120], 'essay,': [2], '..."
7,https://openalex.org/W3023080119,The Superiority of Economists,"{'In': [0], 'this': [1, 120], 'essay,': [2], '..."
10,https://openalex.org/W2526308298,French Connections: The Reception of French So...,"{'Abstract': [0], 'This': [1], 'paper': [2, 48..."
19,https://openalex.org/W3121614287,The Superiority of Economists,"{'In': [0], 'this': [1, 119], 'essay,': [2], '..."
24,https://openalex.org/W2737619627,Iris: A Conversational Agent for Complex Tasks,"{'Today's': [0], 'conversational': [1, 21, 93]..."
26,https://openalex.org/W2055957857,Framing as a Theory of Media Effects,"{'Research': [0], 'on': [1], 'framing': [2, 25..."
30,https://openalex.org/W2005011082,"Community, Communication, and Participation: T...","{'Abstract': [0], 'This': [1], 'study': [2], '..."
32,https://openalex.org/W2161818974,The “Nasty Effect:” Online Incivility and Risk...,"{'Uncivil': [0], 'discourse': [1], 'is': [2], ..."
34,https://openalex.org/W4244993254,Framing as a theory of media effects,"{'Research': [0], 'on': [1], 'framing': [2, 25..."
38,https://openalex.org/W2030626443,Exploring motivations for consumer Web use and...,"{'This': [0], 'study': [1], 'examines': [2], '..."


In [33]:
df2.to_csv("df2.csv", index=False, encoding="utf-8")
df3.to_csv("df3.csv", index=False, encoding="utf-8")

The old code was not paginating (so the dataset achieved during exercise weeks was not fully fleshed out), so a cursor-based pagination fix was attempted, however it was discovered that we had reached OpenAlex API limit, unfortunately too late as well (last day).

Our main strategy for making the code run was to use parallel jobs to save time. Regarding execution time, it did speed up for our sample dataset, yet as mentioned, the full dataset remains to be run. 

The filters set are quite reasonable, as they improve the quality of the dataset, however they may also introduce bias. Recent emerging work may be omitted as well as new fields or interdisciplinary areas that may be underrepresented, simply due to the fact that they have not existed for as long. 